# DSE 200 — Day 1 Lab, Part 2
## Data ETL with Pandas

**Chunk 3.5 · agent off**

Section titles match the slides: **Load, Clean: Missing, Clean: Duplicates, The Ledger, Reshape, Join, Save.**

| Cell | What to do |
| --- | --- |
| `▶ DEMO` | Already written. Run it while Rick talks. |
| `🔍 READ` | **Predict the output before running.** Commit to an answer. |
| `✍️ YOU TRY` | Has a `# TODO`. Yours. |
| `✅ CHECKPOINT` | Run it. It tells you whether you are right. |

**If you get lost:** *Runtime → Run all*. Every section rebuilds its own data, so any section works from cold.

### Manual, on purpose

In the last session today you watch an agent do everything in this notebook in about four minutes.

You do it by hand first, so that when you watch, you know **what it should have checked**.

> **One rule for this whole notebook:** write the row count down after every step.

In [ ]:
# ▶ SETUP — run this first. Finds the data and defines check_read().
import os, textwrap
import pandas as pd

CANDIDATES = ["temperatures.csv", "/content/temperatures.csv",
              "data/temperatures.csv", "../temperatures.csv"]
PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert PATH, ("temperatures.csv not found -- drag it into the Files pane (folder icon, left), "
              "or run: from google.colab import files; files.upload()")

def load():
    """The file, typed and renamed -- the starting point for every section."""
    df = pd.read_csv(PATH, parse_dates=["date"])
    return df.rename(columns={"temperature (C)": "temp"})

def clean():
    """Everything the Clean sections do, in one place -- so later sections start from cold."""
    return load().dropna(subset=["city", "temp"]).drop_duplicates()

_READ = {
"E.1": ("object",
        "read_csv does not guess dates unless you ask. The column arrives as TEXT --\n"
        "'object' is pandas for 'Python strings'. You cannot ask a string what month it is,\n"
        "so parse dates when you load: pd.read_csv(..., parse_dates=['date'])."),
"E.2": ("4090 rows with the subset, 3682 with a plain dropna() -- 408 more gone.",
        "dropna() with no arguments drops a row missing ANY column. 408 readings have a city\n"
        "and a temperature and only lack a date. For 'mean temperature per city' they are\n"
        "perfectly good. The question decides which columns must be present."),
"E.3": ("1.0, 2.0, ... 12.0 -- floats, not integers.",
        "Rows with no date give month = NaN, and NaN is a float. One NaN in a column turns\n"
        "the whole column into floats. Drop the missing dates BEFORE extracting the month."),
"E.4": ("3213 -- all 821 New York readings disappear.",
        "An inner join keeps only rows that match on both sides. 'New York' never matches\n"
        "'New York City', so every New York reading is silently dropped. No error. The result\n"
        "just has four cities in it. Count rows before and after every join."),
}

def check_read(tag, prediction):
    """Reveal the answer to a READ exercise -- after you commit to one."""
    if not str(prediction).strip() or str(prediction).strip() in ("???", "?"):
        print("Write a prediction first. Committing to a wrong answer is how this works.")
        return
    ans, why = _READ[tag]
    print("your prediction:"); print(textwrap.indent(str(prediction), "    "))
    print("\nactual:");        print(textwrap.indent(ans, "    "))
    print("\n" + why)

print("data file:", PATH)
print("check_read() ready.")

## Load

The **E** in ETL — *extract*. Get the file in, and find out what pandas thinks it is.

In [ ]:
# ▶ DEMO — one line in
df = pd.read_csv(PATH)
print(df.shape)
df.head()

In [ ]:
# 🔍 READ E.1 — what type does pandas give the `date` column?
prediction = "???"   # write your answer, then run
check_read("E.1", prediction)

In [ ]:
# ▶ DEMO — check it
df.dtypes

In [ ]:
# ▶ DEMO — fix it on the way in: parse the dates, and give the column a name you can type
df = pd.read_csv(PATH, parse_dates=["date"])
df = df.rename(columns={"temperature (C)": "temp"})
df.dtypes

## Clean: Missing

Count it before you touch it. Then ask whether what is missing is **random** — before you drop anything.

In [ ]:
# ▶ DEMO — three columns, three different holes
df = load()
df.isna().sum()

In [ ]:
# ✍️ YOU TRY — are the nameless rows random?
# If one city's sensor stopped sending its name for a month, dropping the nameless rows
# would remove that month from that city. Check: count the rows with no city, per month.
# Hint: df["date"].dt.month gives the month of each row.

nameless_per_month = None   # TODO: a Series, month -> count of rows with no city

print(nameless_per_month)

In [ ]:
# ✅ CHECKPOINT — nameless rows
assert nameless_per_month is not None, "fill in nameless_per_month"
assert len(nameless_per_month) == 12, "every month should appear"
assert nameless_per_month.min() == 26 and nameless_per_month.max() == 43, nameless_per_month.agg(["min", "max"])
print("Correct -- 26 to 43 in every month. No month is missing, so no city lost a season.")
print("Dropping them is now a decision you can defend, because you looked.")

In [ ]:
# 🔍 READ E.2 — how many rows survive each of these?
#   df.dropna(subset=["city", "temp"])
#   df.dropna()
prediction = "???"
check_read("E.2", prediction)

In [ ]:
# ▶ DEMO — drop only what the question needs, and start the ledger
ledger = [("loaded", len(df))]

df = df.dropna(subset=["city"]);  ledger.append(("no city dropped", len(df)))
df = df.dropna(subset=["temp"]);  ledger.append(("no temperature dropped", len(df)))

for step, rows in ledger:
    print(f"{step:28} {rows:>6,}")

## Clean: Duplicates

Two kinds, and they need **opposite** fixes:

- the same reading **recorded twice** — drop it
- **several readings** in one day — keep them all

In [ ]:
# ▶ DEMO — exact copies: every column identical. Safe to drop.
print(df.duplicated().sum())
df = df.drop_duplicates()
ledger.append(("exact duplicates dropped", len(df)))
len(df)

In [ ]:
# ▶ DEMO — the hard ones: same date, same city, DIFFERENT temperature
repeats = df[df.duplicated(["date", "city"], keep=False)]
repeats.sort_values(["date", "city"]).head()

In [ ]:
# ✍️ YOU TRY — how big is this?
# 1. n_pairs  -- how many (date, city) pairs have MORE than one reading?
# 2. most     -- the largest number of readings for a single (date, city) pair
# Ignore rows with no date. Hint: groupby(["date", "city"]).size()

n_pairs = 0   # TODO
most = 0      # TODO

print(n_pairs, most)

In [ ]:
# ✅ CHECKPOINT — repeated pairs
assert n_pairs == 1089, n_pairs
assert most == 9, most
print("Correct -- 1,089 city-days have several readings, one of them nine.")
print("They differ, so they are not copies. We keep them -- and remember how wild they are:")
print("one New York day runs from 1.3 to 39.2 C.")

## The Ledger

Every cleaning step removed rows. Say how many, and why.

In [ ]:
# ▶ DEMO — what cleaning cost
for step, rows in ledger:
    print(f"{step:28} {rows:>6,}")
print(f"\n{ledger[0][1] - ledger[-1][1]:,} rows gone -- nearly one in five.")

In [ ]:
# ✍️ YOU TRY — did cleaning change the answer?
# 1. raw_seattle   -- mean Seattle temperature straight from the RAW file (pd.read_csv(PATH))
# 2. clean_seattle -- mean Seattle temperature from your cleaned df
# Round both to 2 places.

raw_seattle = 0.0     # TODO
clean_seattle = 0.0   # TODO

print(raw_seattle, clean_seattle)

In [ ]:
# ✅ CHECKPOINT — the Seattle answer
assert raw_seattle == 19.34, raw_seattle
assert clean_seattle == 19.38, clean_seattle
print("Correct -- it barely moved. THIS time.")
print("You only know it barely moved because you counted.")

## Reshape

The **T** in ETL — *transform*. Our file is **long**: one row per reading. A person reading a report wants it **wide**: one row per month, one column per city.

In [ ]:
# 🔍 READ E.3 — this skips dropping the missing dates first. What do the months look like?
#   df.assign(month=df["date"].dt.month)["month"].unique()
prediction = "???"
check_read("E.3", prediction)

In [ ]:
# ▶ DEMO — long to wide
df = clean()
monthly = df.dropna(subset=["date"])
monthly = monthly.assign(month=monthly["date"].dt.month)
wide = monthly.pivot_table(index="month", columns="city",
                           values="temp", aggfunc="mean").round(1)
wide

In [ ]:
# ✍️ YOU TRY — read the grid
# 1. warmest_in_jan  -- the city with the highest mean in January (month 1)
# 2. coldest_in_jul  -- the city with the lowest mean in July (month 7)
# 3. seattle_best    -- the month in which Seattle is warmest
# Hint: wide.loc[1] is January's row. .idxmax() / .idxmin() give the label, not the value.

warmest_in_jan = None   # TODO
coldest_in_jul = None   # TODO
seattle_best = None     # TODO

print(warmest_in_jan, coldest_in_jul, seattle_best)

In [ ]:
# ✅ CHECKPOINT — the grid
assert warmest_in_jan == "Seattle", warmest_in_jan
assert coldest_in_jul == "San Diego", coldest_in_jul
assert seattle_best == 1, seattle_best
print("Correct. Seattle warmest in January. San Diego coldest in July.")
print("The shuffle test in 3.18 said the CITY labels carry nothing.")
print("This grid says the MONTHS carry nothing either. The file is synthetic.")

In [ ]:
# ▶ DEMO — wide back to long. melt is the undo of pivot.
long = wide.reset_index().melt(id_vars="month", var_name="city", value_name="mean_temp")
print(long.shape)    # 12 months x 5 cities
long.head()

## Join

One table was never going to be enough. Here is a second one — and a join that fails without telling you.

In [ ]:
# ▶ DEMO — a second table, and a LEFT join: keep every reading
df = clean()
stations = pd.DataFrame({
    "city":  ["Seattle", "Austin", "Phoenix", "San Diego", "New York City"],
    "state": ["WA", "TX", "AZ", "CA", "NY"],
})
joined = df.merge(stations, on="city", how="left")
print("before:", len(df), " after:", len(joined))
print("readings with no state:", joined["state"].isna().sum())

In [ ]:
# 🔍 READ E.4 — how many rows would an INNER join keep?
#   df.merge(stations, on="city", how="inner")
prediction = "???"
check_read("E.4", prediction)

In [ ]:
# ✍️ YOU TRY — find the city that failed to match, and fix the stations table
# 1. unmatched -- the list of cities in `joined` that got no state
# 2. then fix `stations` so every reading gets a state, and redo the join as `fixed`

unmatched = []   # TODO
fixed = joined   # TODO: fix stations, then join again

print(unmatched)
print(len(fixed), fixed["state"].isna().sum())

In [ ]:
# ✅ CHECKPOINT — the join
assert unmatched == ["New York"], unmatched
assert len(fixed) == len(df) == 4034, len(fixed)
assert fixed["state"].isna().sum() == 0, "some readings still have no state"
print("Correct -- same row count before and after, and every reading has a state.")
print("A join matches EXACTLY. Count rows before and after every one.")

## Save

The **L** in ETL — *load* the result somewhere. Never clean the raw file in place.

In [ ]:
# ▶ DEMO — write it down, two ways
fixed.to_csv("temperatures_clean.csv", index=False)
back = pd.read_csv("temperatures_clean.csv")
print("from CSV:    ", back["date"].dtype)      # the date type is gone

try:
    fixed.to_parquet("temperatures_clean.parquet")
    back = pd.read_parquet("temperatures_clean.parquet")
    print("from Parquet:", back["date"].dtype)  # remembered
except ImportError:
    print("Parquet needs pyarrow -- it is preinstalled on Colab.")

## Your Turn — the whole pipeline

Write one function that does the ETL from the raw file, **and returns the ledger with it**.

In [ ]:
# ✍️ YOU TRY — etl()
# Return (df, ledger):
#   df     -- loaded with dates parsed, temp renamed, no city, no temp and exact copies
#             dropped, joined to the FIXED stations table
#   ledger -- a list of (step, rows) pairs, one per step, starting with ("loaded", 5000)

def etl(path):
    ledger = []
    # TODO
    return None, ledger

result, steps = etl(PATH)
for step, rows in steps:
    print(f"{step:28} {rows:>6,}")

In [ ]:
# ✅ CHECKPOINT — etl()
assert result is not None, "etl() must return the cleaned DataFrame"
rows = [r for _, r in steps]
assert rows[0] == 5000, rows
assert rows[-1] == 4034, rows
assert rows[-1] == rows[-2], "the join changed the row count"
assert result["state"].notna().all(), "some readings have no state"
assert str(result["date"].dtype).startswith("datetime64"), result["date"].dtype
print("Correct -- 5,000 in, 4,034 out, every step accounted for.")

## Wrap

You can now:

- Load a CSV and check what pandas **thinks** each column is
- Drop missing values for a reason — and only the ones the question needs
- Tell a copy from a second reading
- Keep a row ledger
- Reshape long to wide and back
- Join two tables and prove no row was lost

**Next — the last session:** an agent does this whole notebook in four minutes. Watch for four things:

1. Did it **count rows**?
2. Did it **say what it dropped**?
3. Did it **notice the repeated readings** — and decide what they were?
4. Did it **ask whether the answer means anything**?